In [ ]:
# ============================================================
# ROUNDING VS TRUNCATION QUANTIZATION NOISE
# ============================================================
#
# This notebook compares three quantization-error models:
#
#   1. Rounding
#   2. Two's-complement truncation
#   3. Magnitude truncation
#
# The comparison is made in terms of:
#
#   - probability density function (pdf),
#   - mean value,
#   - variance,
#   - simulated quantization-error samples,
#   - histogram of simulated errors.
#
#
# ============================================================
# QUANTIZATION STEP
# ============================================================
#
# For K fractional bits,
#
#                   Delta = 2^(-K).
#
# Therefore, increasing K decreases the quantization step and
# reduces the absolute magnitude of the quantization error.
#
#
# ============================================================
# ROUNDING
# ============================================================
#
# For rounding:
#
#               -Delta/2 <= e <= Delta/2
#
#               m_e = 0
#
#               sigma_e^2 = Delta^2 / 12.
#
#
# ============================================================
# TWO'S-COMPLEMENT TRUNCATION
# ============================================================
#
# For two's-complement truncation:
#
#               -Delta <= e <= 0
#
#               m_e = -Delta/2
#
#               sigma_e^2 = Delta^2 / 12.
#
# This operation therefore introduces a negative bias.
#
#
# ============================================================
# MAGNITUDE TRUNCATION
# ============================================================
#
# For magnitude truncation, assuming equally likely positive and
# negative input values:
#
#               -Delta <= e <= Delta
#
#               m_e = 0
#
#               sigma_e^2 = Delta^2 / 3.
#
# Thus, magnitude truncation has four times the variance of
# rounding:
#
#               (Delta^2/3) / (Delta^2/12) = 4.
#
#
# ============================================================
# NORMALIZED PDF COMPARISON
# ============================================================
#
# The PDF comparison uses the normalized error
#
#                   u = e / Delta.
#
# Normalization is useful here because it allows the SHAPES of the
# three probability distributions to be compared independently of
# the actual value of Delta.
#
# Therefore:
#
#   Rounding:
#       -1/2 <= u <= +1/2
#
#   Two's-complement truncation:
#       -1 <= u <= 0
#
#   Magnitude truncation:
#       -1 <= u <= +1.
#
# Some parts of the probability-density functions overlap.
# Different line styles are therefore used:
#
#   Rounding:
#       solid line
#
#   Two's-complement truncation:
#       dashed line
#
#   Magnitude truncation:
#       dash-dot line.
#
#
# ============================================================
# SIMULATED HISTOGRAM
# ============================================================
#
# Unlike the normalized PDF comparison, the simulated histogram
# is displayed using the ACTUAL quantization error e.
#
# This is important because the effect of the word length K then
# becomes directly visible.
#
# Since
#
#                   Delta = 2^(-K),
#
# increasing K causes the histogram to contract around zero.
#
# For example, with rounding:
#
#       K = 2:
#           Delta = 0.25
#           -0.125 <= e <= 0.125
#
#       K = 4:
#           Delta = 0.0625
#           -0.03125 <= e <= 0.03125
#
#       K = 8:
#           Delta = 0.00390625
#           -0.001953125 <= e <= 0.001953125.
#
# Thus the histogram provides a direct visual demonstration of
# how increasing the number of fractional bits reduces the
# magnitude of the quantization error.
#
#
# ============================================================
# IMPORTANT NOTE
# ============================================================
#
# The variance formulas used here refer to ONE quantizer noise
# source.
#
# Factors such as 2, 5 or (M + N + 1) arise later when independent
# quantization-noise sources are combined at system level.
#
# ============================================================


%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import RadioButtons, IntSlider, VBox, HBox, HTML, Layout, interactive_output
from IPython.display import display


# ------------------------------------------------------------
# Global notebook style
# ------------------------------------------------------------

style_html = HTML("""
<style>

.qt-root {
    font-family: monospace;
    width: 960px;
    max-width: 960px;
}

.qt-description {
    font-size: 13px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #bfc7d5;
    border-left: 6px solid #4a6fa5;
    background: #f7f9fc;
    border-radius: 8px;
    margin-bottom: 8px;
    box-sizing: border-box;
}

.qt-interpretation {
    font-size: 12.5px;
    line-height: 1.45;
    padding: 10px 12px;
    border: 1px solid #d5c58a;
    border-left: 6px solid #b8860b;
    background: #fffaf0;
    border-radius: 8px;
    margin-top: 8px;
    box-sizing: border-box;
}

.qt-interpretation-title {
    font-size: 13px;
    font-weight: bold;
    color: #8a6500;
    margin-bottom: 5px;
}

.qt-box {
    border: 1px solid #c8d0dc;
    border-radius: 9px;
    padding: 9px 12px;
    box-sizing: border-box;
}

.qt-title {
    font-size: 16px;
    font-weight: bold;
    color: #243447;
    margin-bottom: 6px;
}

.qt-info {
    font-size: 13px;
    line-height: 1.52;
}

.qt-label {
    display: inline-block;
    min-width: 225px;
    font-weight: bold;
}

.qt-value {
    font-size: 14px;
    font-weight: bold;
}

.qt-note {
    font-size: 12px;
    line-height: 1.35;
    color: #555555;
    margin-top: 6px;
}

.output_scroll {
    height: unset !important;
    border-radius: 0 !important;
    box-shadow: none !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea {
    overflow-x: visible !important;
    max-width: none !important;
}

</style>
""")


# ------------------------------------------------------------
# Probability-density functions in normalized coordinates
# ------------------------------------------------------------

def rounding_pdf_normalized(u):

    return np.where(
        (u >= -0.5) & (u <= 0.5),
        1.0,
        0.0
    )


def twos_trunc_pdf_normalized(u):

    return np.where(
        (u >= -1.0) & (u <= 0.0),
        1.0,
        0.0
    )


def magnitude_trunc_pdf_normalized(u):

    return np.where(
        (u >= -1.0) & (u <= 1.0),
        0.5,
        0.0
    )


# ------------------------------------------------------------
# Probability-density functions in actual error coordinates
# ------------------------------------------------------------

def rounding_pdf_actual(e, Delta):

    return np.where(
        (e >= -Delta / 2.0) & (e <= Delta / 2.0),
        1.0 / Delta,
        0.0
    )


def twos_trunc_pdf_actual(e, Delta):

    return np.where(
        (e >= -Delta) & (e <= 0.0),
        1.0 / Delta,
        0.0
    )


def magnitude_trunc_pdf_actual(e, Delta):

    return np.where(
        (e >= -Delta) & (e <= Delta),
        1.0 / (2.0 * Delta),
        0.0
    )


# ------------------------------------------------------------
# Generate simulated quantization errors
# ------------------------------------------------------------

def generate_error_samples(method, Delta, N, seed=0):

    rng = np.random.default_rng(seed)

    if method == 'Rounding':

        e = rng.uniform(
            -Delta / 2.0,
            Delta / 2.0,
            N
        )

        mean_theoretical = 0.0

        variance_theoretical = Delta**2 / 12.0


    elif method == "Two's-complement truncation":

        e = rng.uniform(
            -Delta,
            0.0,
            N
        )

        mean_theoretical = -Delta / 2.0

        variance_theoretical = Delta**2 / 12.0


    else:

        e = rng.uniform(
            -Delta,
            Delta,
            N
        )

        mean_theoretical = 0.0

        variance_theoretical = Delta**2 / 3.0


    return e, mean_theoretical, variance_theoretical


# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

title_html = HTML("""
<div class="qt-root">

    <div style="
        font-family:monospace;
        font-size:22px;
        font-weight:bold;
        margin-bottom:8px;
    ">
        Rounding vs Truncation Quantization Noise
    </div>

</div>
""")


# ------------------------------------------------------------
# Description
# ------------------------------------------------------------

description_html = HTML("""
<div class="qt-root">

    <div class="qt-description">

        This notebook compares the quantization-noise behavior of
        <b>rounding</b>, <b>two's-complement truncation</b> and
        <b>magnitude truncation</b>.<br><br>

        The normalized <b>PDF comparison</b> emphasizes the different
        shapes of the three error distributions, independently of the
        quantization step.<br><br>

        The <b>Simulated histogram</b>, however, uses the actual error
        <b>e</b>. Therefore, increasing the number of fractional bits
        <b>K</b> decreases

        <div style="text-align:center; margin:6px 0;">
            <b>Δ = 2<sup>−K</sup></b>
        </div>

        and visibly contracts the quantization-error distribution around zero.

    </div>


    <div class="qt-interpretation">

        <div class="qt-interpretation-title">
            What to observe when K changes
        </div>

        Increasing <b>K</b> does not change the normalized shapes of the
        error probability-density functions because the normalized horizontal
        variable is <b>e/Δ</b>.<br><br>

        In real error units, however, the support intervals shrink with
        <b>Δ = 2<sup>−K</sup></b>. Therefore, in the simulated histogram,
        increasing <b>K</b> makes the distribution visibly narrower around
        zero.<br><br>

        This illustrates directly that increasing the fractional word length
        reduces the absolute quantization error and its variance.

    </div>

</div>
""")


# ------------------------------------------------------------
# Controls
# ------------------------------------------------------------

slider_layout = Layout(
    width='295px'
)


slider_style = {
    'description_width': '105px'
}


display_selector = RadioButtons(
    options=[
        'PDF comparison',
        'Simulated histogram'
    ],
    value='PDF comparison',
    description='Display:',
    style={'description_width': '65px'},
    layout=Layout(
        width='295px'
    )
)


method_selector = RadioButtons(
    options=[
        'Rounding',
        "Two's-complement truncation",
        'Magnitude truncation'
    ],
    value='Rounding',
    description='Method:',
    style={'description_width': '65px'},
    layout=Layout(
        width='295px'
    )
)


bits_slider = IntSlider(
    value=4,
    min=2,
    max=10,
    step=1,
    description='Bits K:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


samples_slider = IntSlider(
    value=5000,
    min=500,
    max=20000,
    step=500,
    description='Samples N:',
    continuous_update=True,
    style=slider_style,
    layout=slider_layout
)


# ------------------------------------------------------------
# Controls box
# ------------------------------------------------------------

controls_box = VBox(
    [
        HTML("<div class='qt-title'>Controls</div>"),
        display_selector,
        method_selector,
        bits_slider,
        samples_slider
    ],
    layout=Layout(
        width='330px',
        min_width='330px',
        border='1px solid #c8d0dc',
        padding='9px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Dynamic information panel
# ------------------------------------------------------------

info_html = HTML()

info_html.layout = Layout(
    width='620px',
    min_width='620px',
    overflow='visible'
)


# ------------------------------------------------------------
# Main update function
# ------------------------------------------------------------

def update_quantization_noise(display_mode, method, K, N):


    # --------------------------------------------------------
    # Quantization step
    # --------------------------------------------------------

    Delta = 2.0**(-K)


    # --------------------------------------------------------
    # Theoretical statistics
    # --------------------------------------------------------

    if method == 'Rounding':

        mean_theoretical = 0.0

        variance_theoretical = Delta**2 / 12.0

        support_text = f"−{Delta / 2.0:.8f} ≤ e ≤ +{Delta / 2.0:.8f}"


    elif method == "Two's-complement truncation":

        mean_theoretical = -Delta / 2.0

        variance_theoretical = Delta**2 / 12.0

        support_text = f"−{Delta:.8f} ≤ e ≤ 0"


    else:

        mean_theoretical = 0.0

        variance_theoretical = Delta**2 / 3.0

        support_text = f"−{Delta:.8f} ≤ e ≤ +{Delta:.8f}"


    # --------------------------------------------------------
    # Information panel
    # --------------------------------------------------------

    info_html.value = f"""
    <div class="qt-box">

        <div class="qt-title">
            Quantization-Noise Summary
        </div>

        <div class="qt-info">

            <span class="qt-label">Selected display</span>
            {display_mode}
            <br>

            <span class="qt-label">Selected method</span>
            {method}
            <br>

            <span class="qt-label">Fractional bits</span>
            K = <span class="qt-value">{K}</span>
            <br>

            <span class="qt-label">Quantization step</span>
            Δ = 2<sup>−{K}</sup> =
            <span class="qt-value">{Delta:.8f}</span>
            <br>

            <span class="qt-label">Actual error interval</span>
            {support_text}
            <br>

            <span class="qt-label">Theoretical mean</span>
            {mean_theoretical:+.8e}
            <br>

            <span class="qt-label">Theoretical variance</span>
            {variance_theoretical:.8e}
            <br>

            <span class="qt-label">Number of samples</span>
            N = {N}

        </div>

        <div class="qt-note">

            The PDF comparison uses normalized error e/Δ and therefore
            emphasizes distribution shape. The simulated histogram uses the
            actual error e, so the effect of increasing K is directly visible
            as a narrowing of the distribution.

        </div>

    </div>
    """


    # ========================================================
    # PDF COMPARISON
    # ========================================================

    if display_mode == 'PDF comparison':

        u = np.linspace(
            -1.2,
            1.2,
            2000
        )


        pdf_round = rounding_pdf_normalized(
            u
        )


        pdf_twos = twos_trunc_pdf_normalized(
            u
        )


        pdf_mag = magnitude_trunc_pdf_normalized(
            u
        )


        fig, ax = plt.subplots(
            figsize=(11.5, 4.8)
        )


        # Rounding: solid line

        ax.plot(
            u,
            pdf_round,
            linewidth=2.6,
            linestyle='-',
            label='Rounding'
        )


        # Two's-complement truncation: dashed line

        ax.plot(
            u,
            pdf_twos,
            linewidth=2.6,
            linestyle='--',
            label="Two's-complement truncation"
        )


        # Magnitude truncation: dash-dot line

        ax.plot(
            u,
            pdf_mag,
            linewidth=2.6,
            linestyle='-.',
            label='Magnitude truncation'
        )


        ax.set_xlim(
            -1.2,
            1.2
        )


        ax.set_ylim(
            0.0,
            1.2
        )


        ax.set_xlabel(
            'Normalized error e / Δ'
        )


        ax.set_ylabel(
            'Normalized density Δ p(e)'
        )


        ax.set_title(
            'Quantization-Error Probability Density Functions',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.6
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.20),
            ncol=3,
            frameon=False,
            fontsize=9
        )


        plt.subplots_adjust(
            left=0.08,
            right=0.98,
            top=0.90,
            bottom=0.28
        )


        plt.show()

        plt.close(fig)


    # ========================================================
    # SIMULATED HISTOGRAM
    # ========================================================

    else:

        e, mean_theoretical, variance_theoretical = generate_error_samples(
            method,
            Delta,
            N,
            seed=0
        )


        mean_measured = np.mean(
            e
        )


        variance_measured = np.var(
            e
        )


        # ----------------------------------------------------
        # Select actual error range and theoretical pdf
        # ----------------------------------------------------

        if method == 'Rounding':

            e_limit = 0.65 * Delta

            e_axis = np.linspace(
                -e_limit,
                e_limit,
                2000
            )

            pdf_actual = rounding_pdf_actual(
                e_axis,
                Delta
            )


        elif method == "Two's-complement truncation":

            e_axis = np.linspace(
                -1.15 * Delta,
                0.15 * Delta,
                2000
            )

            pdf_actual = twos_trunc_pdf_actual(
                e_axis,
                Delta
            )


        else:

            e_axis = np.linspace(
                -1.15 * Delta,
                1.15 * Delta,
                2000
            )

            pdf_actual = magnitude_trunc_pdf_actual(
                e_axis,
                Delta
            )


        # ----------------------------------------------------
        # Figure
        # ----------------------------------------------------

        fig, ax = plt.subplots(
            figsize=(11.5, 4.8)
        )


        # ----------------------------------------------------
        # Histogram in ACTUAL error units
        # ----------------------------------------------------

        ax.hist(
            e,
            bins=40,
            density=True,
            alpha=0.70,
            label='Simulated histogram'
        )


        # ----------------------------------------------------
        # Theoretical pdf in ACTUAL error units
        # ----------------------------------------------------

        ax.plot(
            e_axis,
            pdf_actual,
            linewidth=2.6,
            linestyle='-',
            label='Theoretical pdf'
        )


        # ----------------------------------------------------
        # Zero-error reference
        # ----------------------------------------------------

        ax.axvline(
            0.0,
            linewidth=1.0,
            linestyle=':',
            label='Zero error'
        )


        # ----------------------------------------------------
        # Axis limits
        # ----------------------------------------------------

        if method == 'Rounding':

            ax.set_xlim(
                -0.65 * Delta,
                0.65 * Delta
            )


        elif method == "Two's-complement truncation":

            ax.set_xlim(
                -1.15 * Delta,
                0.15 * Delta
            )


        else:

            ax.set_xlim(
                -1.15 * Delta,
                1.15 * Delta
            )


        ax.set_xlabel(
            'Actual quantization error e'
        )


        ax.set_ylabel(
            'Probability density p(e)'
        )


        ax.set_title(
            f'Simulated Quantization Error: {method}',
            fontsize=12
        )


        ax.grid(
            True,
            linestyle=':',
            alpha=0.6
        )


        ax.legend(
            loc='upper center',
            bbox_to_anchor=(0.5, -0.20),
            ncol=3,
            frameon=False,
            fontsize=9
        )


        plt.subplots_adjust(
            left=0.08,
            right=0.98,
            top=0.90,
            bottom=0.28
        )


        plt.show()

        plt.close(fig)


        # ----------------------------------------------------
        # Simulation statistics
        # ----------------------------------------------------

        simulation_html = HTML(
            f"""
            <div style="
                font-family:monospace;
                font-size:13px;
                line-height:1.50;
                width:940px;
                box-sizing:border-box;
                margin-top:2px;
                margin-bottom:8px;
                padding:8px 10px;
                border:1px solid #c8d0dc;
                border-radius:8px;
                background:#fbfcfe;
            ">

                <b>Simulation Results</b>
                <br>

                Measured mean =
                {mean_measured:+.8e}
                <br>

                Theoretical mean =
                {mean_theoretical:+.8e}
                <br>

                Measured variance =
                {variance_measured:.8e}
                <br>

                Theoretical variance =
                {variance_theoretical:.8e}

            </div>
            """
        )


        display(
            simulation_html
        )


# ------------------------------------------------------------
# Interactive output
# ------------------------------------------------------------

interactive_plot = interactive_output(
    update_quantization_noise,
    {
        'display_mode': display_selector,
        'method': method_selector,
        'K': bits_slider,
        'N': samples_slider
    }
)


interactive_plot.layout = Layout(
    width='auto',
    overflow='visible'
)


# ------------------------------------------------------------
# Top row
# ------------------------------------------------------------

top_row = HBox(
    [
        info_html,
        controls_box
    ],
    layout=Layout(
        width='960px',
        max_width='960px',
        overflow='visible',
        justify_content='space-between',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Final notebook layout
# ------------------------------------------------------------

main_layout = VBox(
    [
        description_html,
        top_row,
        interactive_plot
    ],
    layout=Layout(
        width='960px',
        overflow='visible',
        align_items='flex-start'
    )
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(style_html)

display(title_html)

display(main_layout)